# Camera Stabilization Demo

This tutorial demonstrates NeoVerse's capability to stabilize shaky camera footage by re-planning the camera trajectory.

**Workflow:** Given a shaky input video, NeoVerse first reconstructs the 3D scene and recovers the original (noisy) camera poses. The trajectory is then smoothed using SLERP for rotations and Gaussian filtering for translations. Finally, the video is re-rendered along the stabilized trajectory, with any newly exposed regions inpainted by the diffusion model.

## Configuration

- `use_lora`: Enable 4-step distilled LoRA for fast inference
- `num_frames`: 81 frames for smooth motion (NeoVerse default)
- `alpha_threshold`: Controls which pixels get repainted by diffusion (1.0 = repaint all masked regions)
- `static_scene`: Set to False for videos with dynamic scenes; True for static scenes
- `resize_mode`: "center_crop" preserves aspect ratio while fitting target resolution

In [1]:
import torch
import os
import numpy as np
from torchvision.transforms import functional as F

import sys
sys.path.append("..")
from diffsynth.pipelines.wan_video_neoverse import WanVideoNeoVersePipeline
from diffsynth import save_video
from diffsynth.utils.auxiliary import CameraTrajectory, load_video, homo_matrix_inverse


use_lora = True
model_path = "../models"
reconstructor_path = "../models/NeoVerse/reconstructor.ckpt"
low_vram = False
seed = 42
num_frames = 81
width = 560
height = 336
resize_mode = "center_crop"
static_scene = False
alpha_threshold = 1.0

num_inference_steps = 4 if use_lora else 50
cfg_scale = 1.0 if use_lora else 5.0

/home/yuxue_yang/miniconda3/envs/neoverse/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/yuxue_yang/miniconda3/envs/neoverse/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
lora_path = os.path.join(
    model_path,
    "NeoVerse/loras/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank64.safetensors"
) if use_lora else None

torch.manual_seed(seed)
np.random.seed(seed)

# Load model
print(f"Loading model from {model_path}...")
pipe = WanVideoNeoVersePipeline.from_pretrained(
    local_model_path=model_path,
    reconstructor_path=reconstructor_path,
    lora_path=lora_path,
    lora_alpha=1.0,
    device="cuda",
    torch_dtype=torch.bfloat16,
    enable_vram_management=low_vram,
)
print("Model loaded!")

# Stabilize a Shaky Video

Reconstruct the 3D scene from the shaky input, smooth the recovered camera trajectory, and re-render the video along the stabilized path.

The `smooth_trajectory_slerp` function below handles the trajectory smoothing:
- **Rotations** are interpolated via SLERP across uniformly sampled keyframes
- **Translations** are smoothed with a 1D Gaussian filter

In [ ]:
def smooth_trajectory_slerp(cam2world, num_keyframes=9, smoothing_sigma=1.0):
    from scipy.ndimage import gaussian_filter1d
    from scipy.spatial.transform import Rotation, Slerp
    N = len(cam2world)
    device = cam2world.device if hasattr(cam2world, 'device') else None

    # Convert to numpy for processing
    if isinstance(cam2world, torch.Tensor):
        cam2world_np = cam2world.cpu().numpy()
    else:
        cam2world_np = cam2world

    # 1. Extract rotations and positions
    positions = cam2world_np[:, :3, 3]  # (N, 3)

    # 2. Select keyframes (uniformly distributed)
    keyframe_indices = np.linspace(0, N-1, num_keyframes, dtype=int)
    key_rotations = Rotation.from_matrix(cam2world_np[keyframe_indices, :3, :3])

    # 3. Smooth rotations via SLERP interpolation
    slerp = Slerp(keyframe_indices, key_rotations)
    frame_indices = np.arange(N)
    smoothed_rotations = slerp(frame_indices)

    # 4. Smooth positions via Gaussian filtering
    smoothed_pos = np.zeros_like(positions)
    for i in range(3):
        smoothed_pos[:, i] = gaussian_filter1d(positions[:, i], sigma=smoothing_sigma)

    # 5. Rebuild transformation matrices
    smoothed_cam2world = np.zeros_like(cam2world_np)
    smoothed_cam2world[:, :3, :3] = smoothed_rotations.as_matrix()
    smoothed_cam2world[:, :3, 3] = smoothed_pos
    smoothed_cam2world[:, 3, 3] = 1.0

    # Convert back to original format
    if isinstance(cam2world, torch.Tensor):
        smoothed_cam2world = torch.from_numpy(smoothed_cam2world).to(dtype=cam2world.dtype, device=device)

    return smoothed_cam2world

In [5]:
input_path = "../examples/videos/shaky_video.mp4"
output_path = "../outputs/shaky_to_smooth.mp4"
prompt = "A smooth video with complete scene content. Inpaint any missing regions or margins naturally to match the surrounding scene."
input_video = load_video(input_path, num_frames,
                    resolution=(width, height),
                    resize_mode=resize_mode,
                    static_scene=static_scene)
cam_traj = CameraTrajectory.from_predefined("static")
static_flag = static_scene

with torch.no_grad():
    device = pipe.device
    height, width = input_video[0].size[1], input_video[0].size[0]
    views = {
        "img": torch.stack([F.to_tensor(image)[None] for image in input_video], dim=1).to(device),
        "is_target": torch.zeros((1, len(input_video)), dtype=torch.bool, device=device),
    }
    if static_flag:
        views["is_static"] = torch.ones((1, len(input_video)), dtype=torch.bool, device=device)
        views["timestamp"] = torch.zeros((1, len(input_video)), dtype=torch.int64, device=device)
    else:
        views["is_static"] = torch.zeros((1, len(input_video)), dtype=torch.bool, device=device)
        views["timestamp"] = torch.arange(0, len(input_video), dtype=torch.int64, device=device).unsqueeze(0)

    # Low-VRAM: load reconstructor to GPU before use
    if pipe.vram_management_enabled:
        pipe.reconstructor.to(device)

    with torch.amp.autocast("cuda", dtype=pipe.torch_dtype):
        predictions = pipe.reconstructor(views, is_inference=True, use_motion=False)

    # Low-VRAM: offload reconstructor back to CPU
    if pipe.vram_management_enabled:
        pipe.reconstructor.cpu()
        torch.cuda.empty_cache()

    gaussians = predictions["splats"]
    K = predictions["rendered_intrinsics"][0]
    input_cam2world = predictions["rendered_extrinsics"][0]
    # Smooth the camera trajectory using SLERP and Gaussian filtering
    input_cam2world = smooth_trajectory_slerp(input_cam2world, num_keyframes=9, smoothing_sigma=1.0)
    timestamps = predictions["rendered_timestamps"][0]

    if static_flag:
        K = K[:1].repeat(len(cam_traj), 1, 1)
        timestamps = timestamps[:1].repeat(len(cam_traj))

    # Apply per-trajectory zoom_ratio
    ratio = torch.linspace(1, cam_traj.zoom_ratio, K.shape[0], device=device)
    K_zoomed = K.clone()
    K_zoomed[:, 0, 0] *= ratio
    K_zoomed[:, 1, 1] *= ratio

    target_cam2world = cam_traj.c2w.to(device)
    if cam_traj.mode == "relative" and not static_flag:
        target_cam2world = input_cam2world @ target_cam2world
    target_world2cam = homo_matrix_inverse(target_cam2world)
    target_rgb, target_depth, target_alpha = pipe.reconstructor.gs_renderer.rasterizer.forward(
        gaussians,
        render_viewmats=[target_world2cam],
        render_Ks=[K_zoomed],
        render_timestamps=[timestamps],
        sh_degree=0, width=width, height=height,
    )
    target_mask = (target_alpha > alpha_threshold).float()
    if cam_traj.use_first_frame:
        target_rgb[0, 0] = views["img"][0, 0].permute(1, 2, 0)
        target_mask[0, 0] = 1.0
    wrapped_data = {
        "source_views": views,
        "target_rgb": target_rgb,
        "target_depth": target_depth,
        "target_mask": target_mask,
        "target_poses": target_cam2world.unsqueeze(0),
        "target_intrs": K_zoomed.unsqueeze(0),
    }
    generated_frames = pipe(
        prompt=prompt,
        negative_prompt="",
        seed=seed, rand_device=pipe.device,
        height=height, width=width, num_frames=len(target_cam2world),
        cfg_scale=cfg_scale, num_inference_steps=num_inference_steps, tiled=False,
        **wrapped_data,
    )
    save_video(generated_frames, output_path, fps=16)
print(f"Done! Output saved to: {output_path}")

100%|██████████| 4/4 [00:08<00:00,  2.19s/it]


Done! Output saved to: ../outputs/shaky_to_smooth.mp4


## Results

The stabilized video is saved to `../outputs/`:

- `shaky_to_smooth.mp4`: The input shaky video re-rendered along a smoothed camera trajectory

Regions that become visible due to the trajectory change are seamlessly inpainted by the diffusion model, producing a stable and visually complete output.